In [1]:
import sys
from pathlib import Path

# add the project root to Python's module search path so that
# the notebook can import the source code from src/.
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Binary Symplectic Representation of the $n$-qubit Pauli Group

This notebook develops the binary symplectic representation of the $n$-qubit Pauli operators modulo phase and applies it to the Steane $[[7,1,3]]$ code. The goal is to recast Pauli commutation, stabilizer checks, and syndrome computation as linear algebra over $\mathbb{F}_2$.

This representation replaces manipulations of tensor-product Pauli operators with binary vector and matrix operations, making the structure of stabilizer codes easier to analyze computationally and connecting it directly to the linear-algebraic language of classical error-correcting codes.

We will first identify Pauli operators modulo phase with binary vectors and then show how the symplectic form records Pauli commutation. We will use this representation to construct the binary check matrix of the Steane code and compute syndromes by matrix multiplication. Finally, the resulting binary-symplectic syndrome calculation will be compared with the Pauli-commutation calculation developed earlier in the project.

## 1. Pauli Group, Phase Quotient, and Binary Coordinates

Recall from the previous notebooks that the $n$-qubit Pauli group consists of tensor products of the single-qubit Pauli operators, together with the overall phases $\pm1$ and $\pm i$.

Throughout this notebook, we use Qiskit's qubit-ordering convention and write

$$
\mathcal{P}_n
=
\left\{
i^\ell P_{n-1}\otimes\cdots\otimes P_0
\;\middle|\;
\ell\in\{0,1,2,3\},
\;
P_j\in\{I,X,Y,Z\}
\text{ for } j=0,\ldots,n-1
\right\}.
$$

Thus, the leftmost tensor factor acts on qubit $n-1$, while the rightmost tensor factor acts on qubit $0$.

Towards defining the binary symplectic representation, it is useful to separate the $X$- and $Z$-components of a Pauli operator. Since $Y=iXZ$, every element of $\mathcal{P}_n$ can be written in the form

$$
i^\ell X^{\mathbf{x}}Z^{\mathbf{z}},
$$

where

$$
\mathbf{x}
=
(x_{n-1},\ldots,x_0),
\qquad
\mathbf{z}
=
(z_{n-1},\ldots,z_0)
\in\mathbb{F}_2^n,
$$

and

$$
X^{\mathbf{x}}
:=
X_{n-1}^{x_{n-1}}\cdots X_0^{x_0},
\qquad
Z^{\mathbf{z}}
:=
Z_{n-1}^{z_{n-1}}\cdots Z_0^{z_0}.
$$

Here, $X_j$ and $Z_j$ denote the corresponding Pauli operators acting on qubit $j$ and as the identity on all other qubits. Thus, $x_j$ records the presence of an $X$-component on qubit $j$, while $z_j$ records the presence of a $Z$-component on qubit $j$.

So the $n$-qubit Pauli group can be expressed equivalently as

$$
\mathcal{P}_n
=
\left\{
i^\ell X^{\mathbf{x}}Z^{\mathbf{z}}
\;\middle|\;
\ell\in\{0,1,2,3\},
\;
\mathbf{x},\mathbf{z}\in\mathbb{F}_2^n
\right\}.
$$

In the syndrome extraction notebook, we also considered the quotient of the Pauli group by its center,

$$
Z(\mathcal{P}_n)
=
\langle iI\rangle
=
\{\pm I,\pm iI\}.
$$

We denote this quotient by

$$
\overline{\mathcal{P}}_n
:=
\mathcal{P}_n/Z(\mathcal{P}_n).
$$

Passing to this quotient removes the overall scalar phase. Consequently, every element of $\overline{\mathcal{P}}_n$ can be written uniquely in the form

$$
[X^{\mathbf{x}}Z^{\mathbf{z}}],
$$

with $\mathbf{x},\mathbf{z}\in\mathbb{F}_2^n$.

Moreover, every nonidentity element of $\overline{\mathcal{P}}_n$ has order $2$. The quotient is also abelian. Indeed, since any two Pauli operators $P,Q\in\mathcal{P}_n$ either commute or anticommute, we have $PQ=\pm QP,$ and so their commutator $PQP^{-1}Q^{-1}$ must  satisfy

$$
PQP^{-1}Q^{-1}
\in
\{I,-I\}
\subseteq
Z(\mathcal{P}_n).
$$

Thus all commutators become trivial after quotienting by the center. Hence, $\overline{\mathcal{P}}_n$ is an elementary abelian $2$-group and therefore carries a natural structure as a vector space over $\mathbb{F}_2$.

Define the *binary symplectic representation*

$$
v:
\overline{\mathcal{P}}_n
\longrightarrow
\mathbb{F}_2^{2n}
$$

by

$$
v\!\left([X^{\mathbf{x}}Z^{\mathbf{z}}]\right)
=
(\mathbf{x}\mid\mathbf{z})
=
(x_{n-1},\ldots,x_0\mid z_{n-1},\ldots,z_0).
$$

Given any $[P]=[X^{\mathbf{x}}Z^{\mathbf{z}}]$ and $[Q]=[X^{\mathbf{x}'}Z^{\mathbf{z}'}]$ in $\overline{\mathcal{P}}_n$, it is straightforward to show that 

$$
v([P][Q])
=
(\mathbf{x}+\mathbf{x}'\mid\mathbf{z}+\mathbf{z}')
=
v([P])+v([Q])
$$

and that $v$ respects the $\mathbb{F}_2$-structure between the two groups. Hence, the map $v$ is an $\mathbb{F}_2$-linear isomorphism. 

The binary vector $(\mathbf{x}\mid\mathbf{z})$ therefore records the $X$- and $Z$-components of a Pauli operator modulo phase. This is the standard binary symplectic representation used in the stabilizer literature. The word *symplectic* anticipates the bilinear form introduced in the next section, which recovers the commutation relations of Pauli representatives; see, for example, Gottesman [1] and Calderbank, Rains, Shor, and Sloane [2].

### Ordering convention

The binary coordinates in the image of $v$ are ordered in the same left-to-right order as a Qiskit Pauli string:

$$
(x_{n-1},\ldots,x_0\mid z_{n-1},\ldots,z_0).
$$

For example, the Qiskit string `IIIXXXX` acts with $X$ on qubits $0,1,2,3$, and therefore

$$
v([\mathrm{IIIXXXX}])
=
(0,0,0,1,1,1,1\mid 0,0,0,0,0,0,0).
$$

We implement the binary symplectic representation below.

In [2]:
import numpy as np


# takes a Pauli string and converts it into a binary numpy array
def pauli_label_to_binary(label: str) -> np.ndarray:

    x_array = np.array(
        [1 if char in {"X", "Y"} else 0 for char in label], dtype=np.uint8
    )

    z_array = np.array(
        [1 if char in {"Y", "Z"} else 0 for char in label], dtype=np.uint8
    )

    array = np.concatenate((x_array, z_array))

    return array


# as a demonstrative example, we consider the binary symplectic representation of X_0
assert np.array_equal(
    pauli_label_to_binary("IIIIIIX"),
    np.array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]),
)
print(pauli_label_to_binary("IIIIIIX"))

[0 0 0 0 0 0 1 0 0 0 0 0 0 0]


## 2. The Symplectic Structure

Although the quotient $\overline{\mathcal{P}}_n$ captures the multiplication of Pauli operators modulo phase, its vector-space structure alone no longer distinguishes commuting from anticommuting Pauli representatives. For example,

$$
[X][Z]=[Z][X]
$$

in $\overline{\mathcal{P}}_1$, even though

$$
XZ=-ZX
$$

in $\mathcal{P}_1$.

This commutation information can be recovered by equipping $\mathbb{F}_2^{2n}$ with the bilinear form $\omega: \mathbb{F}_2^{2n} \times \mathbb{F}_2^{2n} \rightarrow \mathbb{F}_2$ defined by

$$
\omega\left(
(\mathbf{x}\mid\mathbf{z}),
(\mathbf{x}'\mid\mathbf{z}')
\right)
=
\mathbf{x}\cdot\mathbf{z}'
+
\mathbf{z}\cdot\mathbf{x}'.
$$

Here, $\mathbf{a}\cdot\mathbf{b}$ denotes the standard binary dot product on $\mathbb{F}_2^n$, with all arithmetic taken in 
$\mathbb{F}_2$. The bilinear form $\omega$ is commonly referred to as the *binary symplectic form*. It is nondegenerate and alternating, and therefore endows $\mathbb{F}_2^{2n}$ with the structure of a symplectic vector space.

To see how $\omega$ encodes Pauli commutation, choose representatives of the form $P=X^{\mathbf{x}}Z^{\mathbf{z}}$ and $Q=X^{\mathbf{x}'}Z^{\mathbf{z}'}$ for the cosets $[P]$ and $[Q]$ in $\overline{\mathcal{P}}_n$. Using the anticommutation relation between $X$ and $Z$, we have

$$
PQ
=
(-1)^{\mathbf{z}\cdot\mathbf{x}'}
X^{\mathbf{x}+\mathbf{x}'}
Z^{\mathbf{z}+\mathbf{z}'},
$$

while reversing the order gives

$$
QP
=
(-1)^{\mathbf{z}'\cdot\mathbf{x}}
X^{\mathbf{x}+\mathbf{x}'}
Z^{\mathbf{z}+\mathbf{z}'}.
$$

Upon comparing these expressions, we obtain

$$
PQ
=
(-1)^{
\mathbf{x}\cdot\mathbf{z}'
+
\mathbf{z}\cdot\mathbf{x}'
}
QP.
$$

Then since

$$
\omega(v([P]),v([Q]))
=
\mathbf{x}\cdot\mathbf{z}'
+
\mathbf{z}\cdot\mathbf{x}',
$$

it follows that

$$
PQ
=
(-1)^{\omega(v([P]),v([Q]))}QP.
$$

Therefore,

$$
\omega(v([P]),v([Q]))=\begin{cases}
0, & \text{if and only if } PQ=QP,\\
1, & \text{if and only if } PQ=-QP.
\end{cases}
$$

Thus, although passing to $\overline{\mathcal{P}}_n$ makes Pauli multiplication commutative, the symplectic form on $\mathbb{F}_2^{2n}$ retains the commutation information of the original Pauli representatives.

The symplectic form can also be expressed as a matrix product. Towards this end, consider the $2n \times 2n$ matrix

$$
\Lambda
=
\begin{pmatrix}
0&I_n\\
I_n&0
\end{pmatrix} \in M_{2n}(\mathbb{F}_2).
$$

If $u=(\mathbf{x}\mid\mathbf{z}) \in \mathbb{F}_2^{2n},$ then $ u\Lambda=(\mathbf{z}\mid\mathbf{x}),$ so $\Lambda$ exchanges the $X$- and $Z$-coordinate blocks. For $w=(\mathbf{x}'\mid\mathbf{z}') \in \mathbb{F}_2^{2n},$ we then have

$$
u\Lambda w^T
=
(\mathbf{z}\mid\mathbf{x})
\begin{pmatrix}
\mathbf{x}'^T\\
\mathbf{z}'^T
\end{pmatrix}
=
\mathbf{z}\cdot\mathbf{x}'
+
\mathbf{x}\cdot\mathbf{z}'.
$$

Therefore,

$$
\omega(u,w)=u\Lambda w^T.
$$

This matrix form will allow pairwise stabilizer commutation relations and syndrome calculations to be expressed directly as binary matrix operations over $\mathbb{F}_2$.

In [3]:
# we construct Lambda now for use later on
identity = np.eye(7, dtype=np.uint8)
zero7 = np.zeros((7, 7), dtype=np.uint8)

Lambda = np.block([[zero7, identity], [identity, zero7]])

assert Lambda.shape == (14, 14)

## 3. The Steane Code in Binary Symplectic Form

Recall that a CSS code is a stabilizer code whose stabilizer admits an independent generating set consisting entirely of $X$-type and $Z$-type Pauli operators [3].

To describe such a code in the binary symplectic formalism, consider a CSS code with parameters $[[n,k,d]]$ and stabilizer $S$. The supports of its $X$-type and $Z$-type stabilizer generators may be encoded by two binary check matrices

$$
H_X\in\mathbb{F}_2^{r_X\times n}
\quad \text{ and } \quad
H_Z\in\mathbb{F}_2^{r_Z\times n},
$$

respectively. Here, $r_X$ denotes the number of independent $X$-type generators and $r_Z$ the number of independent $Z$-type generators, so that

$$
r_X+r_Z=n-k.
$$

After ordering the $X$-type generators before the $Z$-type generators, the corresponding stabilizer check matrix has the block form

$$
H
=
\begin{pmatrix}
H_X&0\\
0&H_Z
\end{pmatrix}
\in\mathbb{F}_2^{(n-k)\times 2n}.
$$

Thus, if $S_1,\ldots,S_{n-k}$ is the corresponding ordered set of independent stabilizer generators, then

$$
H
=
\begin{pmatrix}
v([S_1])\\
\vdots\\
v([S_{n-k}])
\end{pmatrix}.
$$

Using

$$
\Lambda
=
\begin{pmatrix}
0&I_n\\
I_n&0
\end{pmatrix},
$$

we obtain

$$
H\Lambda H^T
=
\begin{pmatrix}
0&H_XH_Z^T\\
H_ZH_X^T&0
\end{pmatrix}.
$$

Since the $i$-th row of $H$ is $v([S_i])$, the $(i,j)$-entry of $H\Lambda H^T$ is

$$
(H\Lambda H^T)_{ij}
=
v([S_i])\Lambda v([S_j])^T
=
\omega(v([S_i]),v([S_j])).
$$

As shown in the previous section,

$$
\omega(v([S_i]),v([S_j]))=0
$$

if and only if $S_i$ and $S_j$ commute. Consequently, the stabilizer generators commute pairwise if and only if

$$
H\Lambda H^T=0.
$$

For a CSS code, this condition is equivalent to

$$
H_XH_Z^T=0.
$$

The matrices $H_X$ and $H_Z$ may also be viewed as parity-check matrices of classical binary linear codes

$$
C_X=\ker H_X,
\qquad
C_Z=\ker H_Z.
$$

Since

$$
C_X^\perp=\operatorname{row}(H_X),
\qquad
C_Z^\perp=\operatorname{row}(H_Z),
$$

the CSS commutation condition may be reexpressed equivalently as

$$
C_X^\perp\subseteq C_Z,
$$

or,

$$
C_Z^\perp\subseteq C_X.
$$

This classical-code viewpoint will become particularly useful when considering syndrome decoding and CSS qLDPC codes.

We now specialize this construction to the Steane $[[7,1,3]]$ code. Recall the six ordered stabilizer generators used throughout this project:

$$
\begin{aligned}
S_1 &= \mathrm{IIIXXXX},\\
S_2 &= \mathrm{IXXIIXX},\\
S_3 &= \mathrm{XIXIXIX},\\
S_4 &= \mathrm{IIIZZZZ},\\
S_5 &= \mathrm{IZZIIZZ},\\
S_6 &= \mathrm{ZIZIZIZ}.
\end{aligned}
$$

The first three generators are $X$-type, while the last three are $Z$-type. Since our binary coordinates follow the same left-to-right ordering as the Qiskit Pauli strings, their binary representations can be read off directly:

$$
\begin{aligned}
v([S_1])
&=
(0,0,0,1,1,1,1\mid 0,0,0,0,0,0,0),\\
v([S_2])
&=
(0,1,1,0,0,1,1\mid 0,0,0,0,0,0,0),\\
v([S_3])
&=
(1,0,1,0,1,0,1\mid 0,0,0,0,0,0,0),\\
v([S_4])
&=
(0,0,0,0,0,0,0\mid 0,0,0,1,1,1,1),\\
v([S_5])
&=
(0,0,0,0,0,0,0\mid 0,1,1,0,0,1,1),\\
v([S_6])
&=
(0,0,0,0,0,0,0\mid 1,0,1,0,1,0,1).
\end{aligned}
$$

Thus, for the Steane code, one has

$$
H_X
=
H_Z
=
\begin{pmatrix}
0&0&0&1&1&1&1\\
0&1&1&0&0&1&1\\
1&0&1&0&1&0&1
\end{pmatrix},
$$

and so the full stabilizer check matrix is

$$
H
=
\begin{pmatrix}
H_X&0\\
0&H_Z
\end{pmatrix}.
$$

Each row of $H_X=H_Z$ has Hamming weight $4$, and any two distinct rows overlap in exactly two positions. Hence every pair of rows has binary dot product zero, and therefore

$$
H_XH_Z^T=0.
$$

It follows that

$$
H\Lambda H^T=0,
$$

which verifies within the binary symplectic formalism that the six Steane stabilizer generators commute pairwise.

We now construct these matrices from the Steane stabilizers and verify the commutation conditions computationally.

In [4]:
from src.steane import Stabilizers

# construct the binary check matrices H_X and H_Z, and also the stabilizer check matrix H

H_X = np.array(
    [
        pauli_label_to_binary(stabilizer.to_label())[:7]
        for stabilizer in Stabilizers[:3]
    ],
    dtype=np.uint8,
)

H_Z = np.array(
    [
        pauli_label_to_binary(stabilizer.to_label())[7:]
        for stabilizer in Stabilizers[3:]
    ],
    dtype=np.uint8,
)

zero = np.zeros(shape=(3, 7), dtype=np.uint8)

H = np.block([[H_X, zero], [zero, H_Z]])

assert H_X.shape == (3, 7)
assert H_Z.shape == (3, 7)
assert H.shape == (6, 14)

We now verify the commutation matrix identities. The first checks pairwise commutation of all six stabilizer generators, while the second checks the CSS cross-commutation condition between the $X$-type and $Z$-type generators.

In [5]:
commutation_matrix = (H @ Lambda @ H.T) % 2
css_commutation_matrix = (H_X @ H_Z.T) % 2

assert np.array_equal(commutation_matrix, np.zeros((6, 6)))
assert np.array_equal(css_commutation_matrix, np.zeros((3, 3)))

## 4. Syndrome Computation in Binary Form

Let $E\in\mathcal{P}_n$ be a Pauli error and let

$$
e=v([E])\in\mathbb{F}_2^{2n}
$$

be its binary representation.

The syndrome associated with stabilizer generators $S_1,\ldots,S_r$ is

$$
s(E)=H\Lambda e^T.
$$

Indeed, the $i$-th component is

$$
s_i(E)
=
v([S_i])\Lambda e^T
=
\omega(v([S_i]),v([E])).
$$

Therefore,

$$
s_i(E) = \begin{cases}
0 & \text{if and only if } S_iE=ES_i,\\
1 & \text{if and only if } S_iE=-ES_i.
\end{cases}
$$

Thus, the same syndrome previously obtained by checking Pauli commutation can be computed entirely through binary matrix multiplication over $\mathbb{F}_2$.

### Example: the error $Z_0$

Consider

$$
E=Z_0.
$$

Under our ordering convention,

$$
e
=
v([Z_0])
=
(0,0,0,0,0,0,0\mid 0,0,0,0,0,0,1).
$$

For the first stabilizer generator,

$$
v([S_1])
=
(0,0,0,1,1,1,1\mid 0,0,0,0,0,0,0),
$$

so

$$
\omega(v([S_1]),e)
=
(0,0,0,1,1,1,1)
\cdot
(0,0,0,0,0,0,1)
=
1.
$$

Thus, $Z_0$ anticommutes with $S_1$.

The same calculation shows that $Z_0$ anticommutes with the three $X$-type generators $S_1,S_2,S_3$ and commutes with the three $Z$-type generators $S_4,S_5,S_6$. Therefore,

$$
s(Z_0)
=
(1,1,1,0,0,0).
$$

This agrees with the syndrome obtained previously from direct Pauli commutation.

The following code verifies this computationally.

In [6]:
error_label = "IIIIIIZ"  # the error Z_0
e = pauli_label_to_binary(error_label)
s = (H @ Lambda @ e) % 2
assert np.array_equal(s, np.array([1, 1, 1, 0, 0, 0]))

We package this calculation into a function that computes the syndrome of an arbitrary Pauli label with respect to the fixed ordered Steane stabilizer generators.

In [7]:
# uses the binary symplectic formalism to compute the syndrome of a Pauli error
# with respect to the fixed ordered Steane stabilizer generators.
def binary_syndrome(label: str) -> tuple[int, ...]:
    e = pauli_label_to_binary(label)
    syndrome_vector = (H @ Lambda @ e) % 2  # dont need e.T since e is a 1-dim nparray
    return tuple(int(bit) for bit in syndrome_vector)


assert binary_syndrome("IIIIIIZ") == (1, 1, 1, 0, 0, 0)

## 5. Computational Verification

We now perform a systematic verification over all $21$ weight-one Pauli errors

$$
X_j,\qquad Y_j,\qquad Z_j,
\qquad
j=0,\ldots,6.
$$

There are three checks of interest:

First, for each of these 21 errors, we verify that our implementation of the binary symplectic representation $v$ agrees with the binary $X$- and $Z$-component data used internally by Qiskit's `Pauli` class.

Second, for each weight-one Pauli error $E$, we compare the syndrome computed from the binary symplectic formula

$$
s(E)=H\Lambda v([E])^T
$$

with the syndrome obtained directly from Pauli commutation relations.

Finally, we use the binary symplectic syndrome calculation to recover a property already verified earlier in the project: the $21$ weight-one Pauli errors have nonzero and distinct syndromes. Thus,

$$
s(E)\neq 0
$$

for every weight-one Pauli error $E$, and

$$
s(E)\neq s(F)
$$

whenever $E$ and $F$ are distinct weight-one Pauli errors.

The significance of this final check is not that the error set itself has been generated independently, but that the same syndrome structure is recovered using the binary symplectic calculation rather than direct Pauli commutation. These computations therefore verify, for all 21 weight-one Pauli errors, that the Pauli commutation relations underlying syndrome extraction are reproduced exactly by the binary symplectic formalism.

In [8]:
from qiskit.quantum_info import Pauli

from src.steane import syndrome


# helper function to collect all 21 of the weight-one Pauli labels
def single_qubit_pauli_label(pauli: str, qubit: int, n: int = 7) -> str:
    return "I" * (n - 1 - qubit) + pauli + "I" * qubit


assert single_qubit_pauli_label("X", 0) == "IIIIIIX"

# collect all such errors into a list
weight_one_errors = []

for pauli in ("X", "Y", "Z"):
    for i in range(7):
        weight_one_errors.append(single_qubit_pauli_label(pauli, i))

# make sure it's the right size
assert len(weight_one_errors) == 21

# cross-check our binary encoding against Qiskit's internal x/z representation.
for error in weight_one_errors:
    p = Pauli(error)

    qiskit_binary = np.concatenate((p.x[::-1], p.z[::-1]))

    assert np.array_equal(pauli_label_to_binary(error), qiskit_binary)

In [9]:
# Now we check all of the syndromes generated via the binary symplectic formalism against
# the algebraic syndrome function from earlier.
for error in weight_one_errors:
    assert binary_syndrome(error) == syndrome(Pauli(error))

Finally, we verify that the $21$ weight-one Pauli errors produce $21$ distinct, nonzero syndromes.

In [10]:
weight_one_syndromes = [binary_syndrome(error) for error in weight_one_errors]

assert len(weight_one_syndromes) == 21

zero_syndrome = (0, 0, 0, 0, 0, 0)
assert zero_syndrome not in weight_one_syndromes
assert len(weight_one_syndromes) == len(set(weight_one_syndromes))

For completeness, we display the syndrome associated with each weight-one Pauli error.

In [11]:
syndrome_table = {error: binary_syndrome(error) for error in weight_one_errors}
for error, syn in syndrome_table.items():
    print(error, syn)

IIIIIIX (0, 0, 0, 1, 1, 1)
IIIIIXI (0, 0, 0, 1, 1, 0)
IIIIXII (0, 0, 0, 1, 0, 1)
IIIXIII (0, 0, 0, 1, 0, 0)
IIXIIII (0, 0, 0, 0, 1, 1)
IXIIIII (0, 0, 0, 0, 1, 0)
XIIIIII (0, 0, 0, 0, 0, 1)
IIIIIIY (1, 1, 1, 1, 1, 1)
IIIIIYI (1, 1, 0, 1, 1, 0)
IIIIYII (1, 0, 1, 1, 0, 1)
IIIYIII (1, 0, 0, 1, 0, 0)
IIYIIII (0, 1, 1, 0, 1, 1)
IYIIIII (0, 1, 0, 0, 1, 0)
YIIIIII (0, 0, 1, 0, 0, 1)
IIIIIIZ (1, 1, 1, 0, 0, 0)
IIIIIZI (1, 1, 0, 0, 0, 0)
IIIIZII (1, 0, 1, 0, 0, 0)
IIIZIII (1, 0, 0, 0, 0, 0)
IIZIIII (0, 1, 1, 0, 0, 0)
IZIIIII (0, 1, 0, 0, 0, 0)
ZIIIIII (0, 0, 1, 0, 0, 0)


## Summary

The binary symplectic representation identifies the quotient $\overline{\mathcal{P}}_n$ with the $\mathbb{F}_2$-vector space $\mathbb{F}_2^{2n}.$ Under this correspondence, Pauli multiplication modulo phase becomes vector addition, while Pauli commutation is encoded by the symplectic form

$$
\omega\!\left(
(\mathbf{x}\mid\mathbf{z}),
(\mathbf{x}'\mid\mathbf{z}')
\right)
=
\mathbf{x}\cdot\mathbf{z}'
+
\mathbf{z}\cdot\mathbf{x}'.
$$

For a stabilizer check matrix $H$, pairwise commutation is expressed compactly as

$$
H\Lambda H^T=0,
$$

and the syndrome of a Pauli error with binary representation $e$ is

$$
s(E)=H\Lambda e^T.
$$

For a CSS code, the stabilizer check matrix has block form

$$
H
=
\begin{pmatrix}
H_X&0\\
0&H_Z
\end{pmatrix},
$$

and the commutation condition reduces to

$$
H_XH_Z^T=0.
$$

The matrices $H_X$ and $H_Z$ may also be viewed as parity-check matrices of classical binary linear codes

$$
C_X=\ker H_X,
\quad \text{ and } \quad
C_Z=\ker H_Z,
$$

for which the CSS commutation condition is equivalently expressed as $C_X^\perp\subseteq C_Z,$ or, $C_Z^\perp\subseteq C_X.$

For the Steane $[[7,1,3]]$ code considered here, the $X$- and $Z$-type check matrices coincide:

$$
H_X=H_Z.
$$

Their rows have pairwise binary dot product zero, so

$$
H_XH_Z^T=0,
$$

which verifies the pairwise commutation of the Steane stabilizer generators in the binary symplectic formalism.

The binary symplectic formalism therefore provides a direct bridge between the Pauli/stabilizer description of the Steane code and the binary parity-check language of classical coding theory. It also gives an independent algebraic implementation of syndrome computation that agrees with the direct Pauli-commutation calculation developed earlier in the project.

### References

[1] D. Gottesman, *Surviving as a Quantum Computer in a Classical World*, 2026 draft, March 31, 2026. In particular, see Section 3.5.1, “The Symplectic Representation of Paulis,” and Section 6.1.5, “Clifford Group and the Symplectic Group.”

[2] A. R. Calderbank, E. M. Rains, P. W. Shor, and N. J. A. Sloane, “Quantum Error Correction via Codes over GF(4),” *IEEE Transactions on Information Theory*, vol. 44, no. 4, pp. 1369–1387, July 1998. doi: 10.1109/18.681315. Also available as arXiv:quant-ph/9608006.

[3] IBM Quantum Learning, “CSS Codes,” in *Foundations of Quantum Error Correction*, lesson “Quantum Code Constructions.” Available at: https://eu-de.quantum.cloud.ibm.com/learning/en/courses/foundations-of-quantum-error-correction/quantum-code-constructions/css-codes